# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a guideline for loading and exploring the FAIR<sup>2</sup> Mass Spectrometry dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s to understand how the data is structured. All entities are referenced by their `@id` fields.

Below we print all record set, field, and column `@id`s.

In [ ]:
# Helper to pretty print
from pprint import pprint

print("Available record set @id values:")
record_set_ids = [rs['@id'] for rs in metadata.record_set] if hasattr(metadata, 'record_set') else []
for rs in metadata.record_set:
    print(f"  - {rs['@id']} (name: {rs.get('name', 'N/A')})")

# Show all fields for each record set
for rs in metadata.record_set:
    print(f"\nRecordSet: {rs['@id']}")
    if 'field' in rs and rs['field']:
        for field in rs['field']:
            print(f"  Field: {field['@id']}, name: {field.get('name', 'N/A')}")
            if 'column' in field and field['column']:
                for col in field['column']:
                    print(f"    Column: {col['@id']}, name: {col.get('name', 'N/A')}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use record set and field `@id`s from the overview above.

- All record set operations are referenced by their `@id`.
- The example below pulls the main analysis record set into a DataFrame.

In [ ]:
# Extract data from selected record sets
# You can change this list to analyze additional record sets
record_sets_to_load = [rs['@id'] for rs in metadata.record_set]
dataframes = {}

for record_set_id in record_sets_to_load:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"\nLoaded DataFrame for RecordSet {record_set_id} (shape: {df.shape})")
        print(f"Available columns: {df.columns.tolist()}")
        display(df.head())

## 4. Exploratory Data Analysis (EDA)
In this section, we perform typical EDA and data cleaning using field `@id` references.

We'll demonstrate:
- Filtering by numeric fields
- Normalization
- Grouping

Update the variable assignments as needed for different fields/attributes.

In [ ]:
# Example: Pick the main record set (first one)
record_set_id = record_sets_to_load[0]
df = dataframes[record_set_id]

# Find a numeric field using record set schema
# We'll pick the first field of type 'Float' or 'Integer', falling back to any numeric-like column
numeric_field_id = None

record_set_schema = next(rs for rs in metadata.record_set if rs['@id'] == record_set_id)
candidate_fields = [f for f in record_set_schema.get('field', []) if f.get('dataType') in ('schema:Float','schema:Integer','schema:Number')]
if candidate_fields:
    numeric_field_id = candidate_fields[0]['@id']
else:
    # Try to heuristically pick a numeric column
    for col in df.columns:
        if df[col].dtype in ('float64', 'int64'):
            numeric_field_id = col
            break

print(f"Using numeric field for EDA: {numeric_field_id}")

# Filter, normalize, group
if numeric_field_id is not None:
    try:
        # Try conversion if not already numeric
        df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    except Exception:
        pass
    threshold = df[numeric_field_id].mean()  # Can update to any static value
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    display(filtered_df.head())

    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"Normalized field {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by first string/categorical field
    group_field_id = None
    for f in record_set_schema.get('field', []):
        col_id = f['@id'] if f['@id'] in df.columns and df[f['@id']].dtype == 'object' else None
        if col_id:
            group_field_id = col_id
            break
    if group_field_id is not None:
        print(f"\nGrouping by field: {group_field_id}")
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame()
        display(grouped_df.head())
else:
    print("No numeric field identified in this record set.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset using field `@id`s.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize the distribution of the main numeric field
if numeric_field_id is not None and numeric_field_id in df.columns:
    plt.figure(figsize=(8,5))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=30)
    plt.title(f'Distribution of {numeric_field_id} in RecordSet {record_set_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

# If grouping worked, show a bar plot of means by group
if 'grouped_df' in locals() and grouped_df is not None:
    plt.figure(figsize=(10,5))
    grouped_df.plot(kind='bar', legend=False)
    plt.title(f'Mean {numeric_field_id} per {group_field_id}')
    plt.ylabel(f'Mean {numeric_field_id}')
    plt.xlabel(group_field_id)
    plt.show()

## 6. Conclusion
In this notebook, we've demonstrated how to load, inspect, and analyze the FAIR^2 Mass Spectrometry dataset in a reproducible and semantically meaningful way using the `mlcroissant` library.

**Key observations:**
- Data entities are referenced by their `@id` fields for reproducibility.
- Data extraction, transformation, and visualization steps were performed on the identified record sets and fields.
- This approach ensures consistent use of semantic schemas and can be adapted for larger or similar Croissant-based datasets.